In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/heatcheck/raw"
CHECKPOINTS = f"{VOLUME}/_checkpoints"

def load_bronze(source):
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("wholetext", "true")
        .load(f"{VOLUME}/{source}")
        .select(
            F.col("value").alias("raw_json"),
            F.col("_metadata.file_path").alias("source_file"),
            F.current_timestamp().alias("ingested_at"),
        )
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINTS}/{source}")
        .trigger(availableNow=True)
        .toTable(f"workspace.heatcheck.bronze_{source}")
        .awaitTermination())
    print("Loaded", source)

for source in ["wbgt", "air_temperature"]:
    load_bronze(source)

In [0]:
%sql
SELECT source_file, ingested_at FROM workspace.heatcheck.bronze_wbgt